In [1]:
from ast import *
from utils import *

In [2]:
Binding = tuple[Name, expr]
Temporaries = list[Binding]

In [ ]:

def rco_exp(e, need_atomic) -> tuple[e,Temporaries]:
        # YOUR CODE HERE
        match e:
            case Constant(value):
                return (Constant(value), [])
            case Name(id):
                  return (Name(id), [])
            case UnaryOp(USub(),v):
                  new_v , bindings = rco_exp(v,True)
                  new_e = UnaryOp(USub(),new_v)
                  if need_atomic:
                        temp_name = generate_name("temp")
                        return (Name(temp_name), bindings + [(temp_name,new_e)])
                  else:
                        return (new_e, bindings)
            case BinOp(left, Add(),right):
                    new_l , b1 = rco_exp(left,True)
                    new_r, b2 = rco_exp(right,True)
                    new_e = BinOp(new_l, Add(),new_r)
                    if need_atomic:
                          temp_name = generate_name("temp")
                          return (Name(temp_name), b1 + b2 + [(temp_name,new_e)])
                    else:
                          return (new_e, b1 + b2)

            case BinOp(left, Sub(), right):
                    new_l, b1 = rco_exp(left,True)
                    new_r, b2 = rco_exp(right,True)
                    new_e = BinOp(new_l, Sub(), new_r)
                    if need_atomic:
                          temp_name = generate_name("temp")
                          return (Name(temp_name), b1 + b2 + [(temp_name,new_e)])
                    else:
                          return (new_e , b1 + b2)

            case Call(Name('input_int'),[]):
                    new_e = Call(Name('input_int'),[])
                    if need_atomic:
                          temp_name = generate_name("temp")
                          return (Name(temp_name), [(temp_name,new_e)])
                    else:
                          return (new_e, [])
                    
              
                
       
            
        pass 

In [4]:
code = """
x = (4+5) + (7-2)
print(x)
"""
parsed_code = parse(code)

In [5]:
print(dump(parsed_code,indent=4))

Module(
    body=[
        Assign(
            targets=[
                Name(id='x', ctx=Store())],
            value=BinOp(
                left=BinOp(
                    left=Constant(value=4),
                    op=Add(),
                    right=Constant(value=5)),
                op=Add(),
                right=BinOp(
                    left=Constant(value=7),
                    op=Sub(),
                    right=Constant(value=2)))),
        Expr(
            value=Call(
                func=Name(id='print', ctx=Load()),
                args=[
                    Name(id='x', ctx=Load())]))])


In [ ]:
def rco_stmt(s: stmt) -> List[stmt]:
        # YOUR CODE HERE
        match s:

            case Assign([Name(var)],v):
                value, bindings = rco_exp(v,False)
                stmts = [Assign([Name(t)],e) for (t,e) in bindings]
                return stmts + [Assign([Name(var)], value)]
             
            case Expr(Call(Name('print'),[arg])):
                value, bindings = rco_exp(arg,True)
                stmts = [Assign([Name(t)],e) for (t,e) in bindings]
                return stmts + [Expr(Call(Name('print'),[value]))]
             
            case Expr(exp):
                value, bindings = rco_exp(exp,False)
                stmts = [Assign([Name(t)],e) for (t,e) in bindings]
                return stmts + [Expr(value)]
       
          

        pass        

In [7]:
def remove_complex_operands(p: Module) -> Module:
        # YOUR CODE HERE
        match p:
            case Module(body):
                new_body = []
                for stmt in body:
                     new_body.extend(rco_stmt(stmt))
                return Module(new_body)
        pass         

In [8]:
aagh = remove_complex_operands(parsed_code)

In [9]:
print(dump(aagh,indent=4))

Module(
    body=[
        Assign(
            targets=[
                Name(id='temp.0', ctx=Load())],
            value=BinOp(
                left=Constant(value=4),
                op=Add(),
                right=Constant(value=5))),
        Assign(
            targets=[
                Name(id='temp.1', ctx=Load())],
            value=BinOp(
                left=Constant(value=7),
                op=Sub(),
                right=Constant(value=2))),
        Assign(
            targets=Name(id='x', ctx=Load()),
            value=BinOp(
                left=Name(id='temp.0', ctx=Load()),
                op=Add(),
                right=Name(id='temp.1', ctx=Load()))),
        Expr(
            value=Call(
                func=Name(id='print', ctx=Load()),
                args=[
                    Name(id='x', ctx=Load())]))])
